In [1]:
import math
import numpy as np
import pandas as pd

print("Libraries imported successfully! ✅")


Libraries imported successfully! ✅


In [2]:
df = pd.read_csv("../data/processed/manali_hybrid_scores.csv")

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (20, 29)


,place_id,name,address,country,latitude,longitude,rating,reviews,category,data_quality_score,...,interest_count,structured_score,recommendation_text,tfidf_score,semantic_text,semantic_score,rating_score,log_reviews,popularity_score,hybrid_score
0,ChIJPeVowAaIBDkRnIfuPkskqi0,Hadimba Devi Temple,"Regency Road, Siyal Rd, Siyal, Manali, Himacha...",India,32.248353,77.181573,4.6,49688,Tourist attraction,6,...,6,0.600099,Hadimba Devi Temple Tourist attraction culture...,0.0,Hadimba Devi Temple. Tourist attraction. cultu...,0.321686,0.777778,10.813539,1.000000,0.463197
1,ChIJ3VQyL0WHBDkR4Uml16nJto8,Old Manali snow point,"65XJ+J2W, Hadimba Temple Path, Old Manali, Man...",India,32.249112,77.180076,4.6,428,Tourist attraction,6,...,5,0.717137,Old Manali snow point Tourist attraction cultu...,0.0,Old Manali snow point. Tourist attraction. cul...,0.374756,0.777778,6.061457,0.429428,0.451321
2,ChIJk9K3np2HBDkRS4FxsgYCMKI,Nehru Kund,"Bashisht, Himachal Pradesh 175103, India",India,32.285982,77.179824,4.4,7767,Tourist attraction,6,...,2,0.472456,"Nehru Kund Tourist attraction history, photogr...",0.0,"Nehru Kund. Tourist attraction. history, photo...",0.417762,0.555556,8.957768,0.777182,0.404494
3,ChIJJSBNlWOJBDkRkmyIuy0OvGE,Kullu Manali River rafting,"65VQ+7MF, Siyal, Manali, Himachal Pradesh 1751...",India,32.243187,77.189176,4.5,88,Tourist attraction,6,...,0,0.000000,Kullu Manali River rafting Tourist attraction,0.0,Kullu Manali River rafting. Tourist attraction.,0.315588,0.666667,4.488636,0.240583,0.218735
4,ChIJHZ3eboyHBDkRBLrRpkcXmO4,Jogini Falls,"On water fall way V.P.O.-Vashist 5 km.from, Ma...",India,32.275076,77.188146,4.6,10842,Tourist attraction,6,...,4,0.734968,"Jogini Falls Tourist attraction family, histor...",0.0,"Jogini Falls. Tourist attraction. family, hist...",0.418287,0.777778,9.291275,0.817225,0.507617


In [3]:
df[[
    "name",
    "latitude",
    "longitude",
    "hybrid_score"
]].head(10)


,name,latitude,longitude,hybrid_score
0,Hadimba Devi Temple,32.248353,77.181573,0.463197
1,Old Manali snow point,32.249112,77.180076,0.451321
2,Nehru Kund,32.285982,77.179824,0.404494
3,Kullu Manali River rafting,32.243187,77.189176,0.218735
4,Jogini Falls,32.275076,77.188146,0.507617
5,Van Vihar National Park,32.239113,77.189089,0.447468
6,Manali View Point,32.233835,77.187359,0.468197
7,Rahala Waterfalls,32.336362,77.218365,0.343484
8,Lama Dugh Trek Start Point,32.248903,77.175013,0.423811
9,Atal Bihari statue,32.245671,77.189757,0.202767


In [4]:
def estimate_visit_minutes(name):
    name_lower = str(name).lower()

    if "waterfall" in name_lower or "falls" in name_lower:
        return 90

    if "temple" in name_lower:
        return 60

    if "view point" in name_lower or "viewpoint" in name_lower:
        return 45

    if "trek" in name_lower:
        return 150

    if "national park" in name_lower:
        return 120

    if "bazaar" in name_lower:
        return 90

    if "snow point" in name_lower:
        return 90

    if "igloo" in name_lower:
        return 90

    return 60


df["visit_minutes"] = df["name"].apply(
    estimate_visit_minutes
)

df[[
    "name",
    "visit_minutes"
]]


,name,visit_minutes
0,Hadimba Devi Temple,60
1,Old Manali snow point,90
2,Nehru Kund,60
3,Kullu Manali River rafting,60
4,Jogini Falls,90
5,Van Vihar National Park,120
6,Manali View Point,45
7,Rahala Waterfalls,90
8,Lama Dugh Trek Start Point,150
9,Atal Bihari statue,60


In [5]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(
        math.sqrt(a),
        math.sqrt(1 - a)
    )

    return R * c


In [6]:
place_a = df.iloc[0]
place_b = df.iloc[1]

distance = haversine_km(
    place_a["latitude"],
    place_a["longitude"],
    place_b["latitude"],
    place_b["longitude"]
)

print(
    f"Distance between {place_a['name']} and "
    f"{place_b['name']}: {distance:.2f} km"
)


Distance between Hadimba Devi Temple and Old Manali snow point: 0.16 km


In [7]:
AVERAGE_SPEED_KMPH = 25

def travel_minutes(distance_km):
    return (distance_km / AVERAGE_SPEED_KMPH) * 60

print(
    "Example travel time:",
    round(travel_minutes(distance), 1),
    "minutes"
)


Example travel time: 0.4 minutes


In [8]:
coordinates = df[
    ["latitude", "longitude"]
].to_numpy()

n = len(df)

distance_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        distance_matrix[i, j] = haversine_km(
            coordinates[i][0],
            coordinates[i][1],
            coordinates[j][0],
            coordinates[j][1]
        )

print("Distance matrix shape:", distance_matrix.shape)


Distance matrix shape: (20, 20)


In [9]:
travel_time_matrix = (
    distance_matrix / AVERAGE_SPEED_KMPH
) * 60

print("Travel-time matrix shape:", travel_time_matrix.shape)


Travel-time matrix shape: (20, 20)


In [10]:
def select_candidates(top_n=10):
    return (
        df.sort_values(
            "hybrid_score",
            ascending=False
        )
        .head(top_n)
        .copy()
    )

candidates = select_candidates(top_n=10)

candidates[[
    "name",
    "hybrid_score",
    "rating",
    "reviews",
    "visit_minutes"
]]


,name,hybrid_score,rating,reviews,visit_minutes
4,Jogini Falls,0.507617,4.6,10842,90
6,Manali View Point,0.468197,4.6,87,45
0,Hadimba Devi Temple,0.463197,4.6,49688,60
1,Old Manali snow point,0.451321,4.6,428,90
5,Van Vihar National Park,0.447468,4.2,9050,120
8,Lama Dugh Trek Start Point,0.423811,4.6,297,150
10,Kharma valley,0.406514,4.8,143,60
2,Nehru Kund,0.404494,4.4,7767,60
14,Baror Parsha Waterfall,0.397755,4.7,489,90
18,Gulaba Viewpoint,0.397028,4.5,3576,45


In [11]:
NUM_DAYS = 3
DAY_START_HOUR = 9
MAX_DAY_MINUTES = 8 * 60

print("Days:", NUM_DAYS)
print("Start:", f"{DAY_START_HOUR}:00")
print("Available minutes/day:", MAX_DAY_MINUTES)


Days: 3
Start: 9:00
Available minutes/day: 480


In [12]:
def build_daily_route(
    candidates,
    day_start_index,
    available_minutes=MAX_DAY_MINUTES
):
    selected = []
    remaining = set(candidates.index)

    current_index = day_start_index
    used_minutes = 0

    while remaining:
        feasible = []

        for idx in remaining:
            travel = travel_time_matrix[
                current_index,
                idx
            ]

            visit = candidates.loc[
                idx,
                "visit_minutes"
            ]

            total_time = travel + visit

            if used_minutes + total_time <= available_minutes:
                feasible.append(
                    (idx, travel, visit)
                )

        if not feasible:
            break

        best_idx = max(
            feasible,
            key=lambda item: (
                candidates.loc[item[0], "hybrid_score"]
                / (1 + item[1])
            )
        )[0]

        travel = travel_time_matrix[
            current_index,
            best_idx
        ]

        visit = candidates.loc[
            best_idx,
            "visit_minutes"
        ]

        selected.append(
            {
                "index": best_idx,
                "travel_minutes": travel,
                "visit_minutes": visit
            }
        )

        used_minutes += travel + visit
        remaining.remove(best_idx)
        current_index = best_idx

    return selected


In [13]:
def format_time(total_minutes):
    hours = int(total_minutes // 60)
    minutes = int(total_minutes % 60)

    suffix = "AM" if hours < 12 else "PM"

    display_hour = hours % 12
    if display_hour == 0:
        display_hour = 12

    return f"{display_hour}:{minutes:02d} {suffix}"


In [14]:
first_index = candidates.index[0]

day_route = build_daily_route(
    candidates,
    first_index
)

current_time = DAY_START_HOUR * 60

for stop_number, stop in enumerate(day_route, start=1):
    arrival = current_time + stop["travel_minutes"]
    departure = arrival + stop["visit_minutes"]

    idx = stop["index"]

    print(
        f"{stop_number}. "
        f"{format_time(arrival)} → "
        f"{format_time(departure)} | "
        f"{candidates.loc[idx, 'name']}"
    )

    current_time = departure


1. 9:00 AM → 10:30 AM | Jogini Falls
2. 10:33 AM → 11:33 AM | Nehru Kund
3. 11:43 AM → 12:43 PM | Hadimba Devi Temple
4. 12:43 PM → 2:13 PM | Old Manali snow point
5. 2:15 PM → 4:45 PM | Lama Dugh Trek Start Point


In [15]:
def generate_itinerary(
    candidates,
    num_days=3,
    max_day_minutes=480
):
    remaining = candidates.copy()
    itinerary = []

    for day in range(1, num_days + 1):

        if remaining.empty:
            break

        # Start each day at the highest-ranked remaining place.
        start_index = remaining.index[0]

        route = build_daily_route(
            remaining,
            start_index,
            available_minutes=max_day_minutes
        )

        if not route:
            break

        day_stops = []

        current_time = DAY_START_HOUR * 60

        for stop in route:
            idx = stop["index"]

            arrival = (
                current_time
                + stop["travel_minutes"]
            )

            departure = (
                arrival
                + stop["visit_minutes"]
            )

            day_stops.append({
                "place": remaining.loc[idx, "name"],
                "arrival": format_time(arrival),
                "departure": format_time(departure),
                "travel_before_minutes":
                    round(stop["travel_minutes"], 1),
                "visit_minutes":
                    stop["visit_minutes"],
                "hybrid_score":
                    remaining.loc[idx, "hybrid_score"]
            })

            current_time = departure

        itinerary.append({
            "day": day,
            "stops": day_stops
        })

        used_indices = [
            stop["index"]
            for stop in route
        ]

        remaining = remaining.drop(
            index=used_indices
        )

    return itinerary


In [16]:
itinerary = generate_itinerary(
    candidates,
    num_days=NUM_DAYS,
    max_day_minutes=MAX_DAY_MINUTES
)

for day_plan in itinerary:
    print(f"\n📅 DAY {day_plan['day']}")
    print("-" * 60)

    for stop_number, stop in enumerate(
        day_plan["stops"],
        start=1
    ):
        print(
            f"{stop_number}. "
            f"{stop['arrival']} - "
            f"{stop['departure']} | "
            f"{stop['place']} "
            f"(score={stop['hybrid_score']:.3f})"
        )



📅 DAY 1
------------------------------------------------------------
1. 9:00 AM - 10:30 AM | Jogini Falls (score=0.508)
2. 10:33 AM - 11:33 AM | Nehru Kund (score=0.404)
3. 11:43 AM - 12:43 PM | Hadimba Devi Temple (score=0.463)
4. 12:43 PM - 2:13 PM | Old Manali snow point (score=0.451)
5. 2:15 PM - 4:45 PM | Lama Dugh Trek Start Point (score=0.424)

📅 DAY 2
------------------------------------------------------------
1. 9:00 AM - 9:45 AM | Manali View Point (score=0.468)
2. 9:46 AM - 11:46 AM | Van Vihar National Park (score=0.447)
3. 11:52 AM - 12:52 PM | Kharma valley (score=0.407)
4. 1:06 PM - 2:36 PM | Baror Parsha Waterfall (score=0.398)
5. 3:05 PM - 3:50 PM | Gulaba Viewpoint (score=0.397)


In [17]:
itinerary_rows = []

for day_plan in itinerary:
    for stop_number, stop in enumerate(
        day_plan["stops"],
        start=1
    ):
        itinerary_rows.append({
            "day": day_plan["day"],
            "stop": stop_number,
            "place": stop["place"],
            "arrival": stop["arrival"],
            "departure": stop["departure"],
            "travel_before_minutes":
                stop["travel_before_minutes"],
            "visit_minutes":
                stop["visit_minutes"],
            "hybrid_score":
                stop["hybrid_score"]
        })

itinerary_df = pd.DataFrame(
    itinerary_rows
)

itinerary_df


,day,stop,place,arrival,departure,travel_before_minutes,visit_minutes,hybrid_score
0,1,1,Jogini Falls,9:00 AM,10:30 AM,0.0,90,0.507617
1,1,2,Nehru Kund,10:33 AM,11:33 AM,3.5,60,0.404494
2,1,3,Hadimba Devi Temple,11:43 AM,12:43 PM,10.0,60,0.463197
3,1,4,Old Manali snow point,12:43 PM,2:13 PM,0.4,90,0.451321
4,1,5,Lama Dugh Trek Start Point,2:15 PM,4:45 PM,1.1,150,0.423811
5,2,1,Manali View Point,9:00 AM,9:45 AM,0.0,45,0.468197
6,2,2,Van Vihar National Park,9:46 AM,11:46 AM,1.5,120,0.447468
7,2,3,Kharma valley,11:52 AM,12:52 PM,6.3,60,0.406514
8,2,4,Baror Parsha Waterfall,1:06 PM,2:36 PM,13.4,90,0.397755
9,2,5,Gulaba Viewpoint,3:05 PM,3:50 PM,29.7,45,0.397028


In [18]:
total_visit_minutes = itinerary_df[
    "visit_minutes"
].sum()

total_travel_minutes = itinerary_df[
    "travel_before_minutes"
].sum()

average_score = itinerary_df[
    "hybrid_score"
].mean()

print(
    "Places scheduled:",
    len(itinerary_df)
)

print(
    "Total visit time:",
    round(total_visit_minutes, 1),
    "minutes"
)

print(
    "Estimated travel time:",
    round(total_travel_minutes, 1),
    "minutes"
)

print(
    "Average recommendation score:",
    round(average_score, 3)
)


Places scheduled: 10
Total visit time: 810 minutes
Estimated travel time: 65.9 minutes
Average recommendation score: 0.437


In [19]:
output_path = "../data/processed/manali_itinerary_baseline.csv"

itinerary_df.to_csv(
    output_path,
    index=False
)

print(
    f"✅ Itinerary saved to: {output_path}"
)


✅ Itinerary saved to: ../data/processed/manali_itinerary_baseline.csv
